# QC Overview
Mapping rates, variant counts, and alignment metrics for all runs in the database.
Works for both `bulk` (STAR) and `wes` (BWA-MEM) runs.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import sqlite3, re
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

DB_PATH = 'results/variants.db'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

print('Connected to:', DB_PATH)

## Cross-sample QC overview
Mapping rate and filtered variant counts for every run. Colour by data type (sc vs WES).

In [ ]:
runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt,
           r.sequencing, r.run_date
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY s.cell_line, r.run_id
""")

if runs.empty:
    print('No runs found in database.')
else:
    runs['label'] = runs['cell_line'].fillna(runs['sample'])
    # Infer data type: WES runs have '_wes_' in run_id
    runs['data_type'] = runs['run_id'].apply(
        lambda x: 'WES' if '_wes_' in x.lower() else 'sc/bulk'
    )
    runs_sorted = runs.sort_values('mapping_rate', ascending=True)

    color_map = {'WES': '#6d597a', 'sc/bulk': '#457b9d'}
    map_colors  = [('#e63946' if v < 85 else color_map[dt])
                   for v, dt in zip(runs_sorted['mapping_rate'], runs_sorted['data_type'])]
    runs_sorted2 = runs.sort_values('total_filt', ascending=True)
    var_colors   = [color_map[dt] for dt in runs_sorted2['data_type']]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Mapping rate (%) — all runs', 'Filtered variants per run'),
        horizontal_spacing=0.12
    )
    fig.add_trace(go.Bar(
        y=runs_sorted['label'], x=runs_sorted['mapping_rate'],
        orientation='h', marker_color=map_colors,
        customdata=runs_sorted['data_type'],
        hovertemplate='%{y}  [%{customdata}]<br>Mapping: %{x:.1f}%<extra></extra>'
    ), row=1, col=1)
    fig.add_vline(x=85, line_dash='dash', line_color='#e63946',
                  annotation_text='85%', row=1, col=1)
    fig.add_trace(go.Bar(
        y=runs_sorted2['label'], x=runs_sorted2['total_filt'],
        orientation='h', marker_color=var_colors,
        customdata=runs_sorted2['data_type'],
        hovertemplate='%{y}  [%{customdata}]<br>Filtered variants: %{x:,}<extra></extra>'
    ), row=1, col=2)

    n = len(runs)
    fig.update_layout(
        height=max(400, 22 * n),
        title=f'Run QC overview — {n} runs  (purple = WES · blue = sc/bulk)',
        showlegend=False
    )
    fig.update_xaxes(range=[0, 105], row=1, col=1)
    fig.show()

    summary = (
        runs.groupby(['label', 'data_type']).agg(
            n_runs=('run_id', 'count'),
            mean_mapping=('mapping_rate', 'mean'),
            mean_filt_vars=('total_filt', 'mean'),
        ).reset_index().sort_values('mean_filt_vars', ascending=False)
    )
    display(summary.style.format({'mean_mapping': '{:.1f}', 'mean_filt_vars': '{:,.0f}'}))

## WES vs sc/bulk — side-by-side stats
Highlights the difference in depth and variant count between the two data types.

In [ ]:
if not runs.empty:
    by_type = runs.groupby('data_type').agg(
        n_runs=('run_id', 'count'),
        mean_mapping=('mapping_rate', 'mean'),
        median_filt_vars=('total_filt', 'median'),
        mean_filt_vars=('total_filt', 'mean'),
        min_filt_vars=('total_filt', 'min'),
        max_filt_vars=('total_filt', 'max'),
    ).reset_index()
    display(by_type.style.format({
        'mean_mapping': '{:.2f}',
        'median_filt_vars': '{:,.0f}',
        'mean_filt_vars': '{:,.0f}',
        'min_filt_vars': '{:,.0f}',
        'max_filt_vars': '{:,.0f}',
    }))

    fig2 = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Filtered variant counts by data type', 'Mapping rate by data type'),
        horizontal_spacing=0.14
    )
    for dt, color in color_map.items():
        sub = runs[runs['data_type'] == dt]
        fig2.add_trace(go.Box(
            y=sub['total_filt'], name=dt, marker_color=color,
            boxmean=True, boxpoints='all', jitter=0.4, pointpos=0
        ), row=1, col=1)
        fig2.add_trace(go.Box(
            y=sub['mapping_rate'], name=dt, marker_color=color,
            boxmean=True, boxpoints='all', jitter=0.4, pointpos=0,
            showlegend=False
        ), row=1, col=2)
    fig2.update_yaxes(title_text='Filtered variants', row=1, col=1)
    fig2.update_yaxes(title_text='Mapping rate (%)', row=1, col=2)
    fig2.update_layout(height=420, title='WES vs sc/bulk comparison')
    fig2.show()

## STAR alignment QC (sc/bulk runs only)
Annotated splice junction percentage and short-read fraction from STAR logs.
WES runs (BWA) are excluded — these metrics are RNA-specific.

In [ ]:
def parse_star_log(run_id, sample):
    log_path = f'results/{run_id}/star/{sample}/Log.final.out'
    if not os.path.exists(log_path):
        return None, None
    text = open(log_path).read()
    m_annot = re.search(r'% of splices: Annotated \(sjdb\).*?([\d.]+)%', text)
    m_short = re.search(r'% of reads unmapped: too short.*?([\d.]+)%', text)
    return (float(m_annot.group(1)) if m_annot else None,
            float(m_short.group(1))  if m_short  else None)

sc_runs = runs[runs['data_type'] == 'sc/bulk'].copy() if not runs.empty else pd.DataFrame()

if sc_runs.empty:
    print('No sc/bulk runs found.')
else:
    splice_vals, short_vals = [], []
    for _, row in sc_runs.iterrows():
        sp, sh = parse_star_log(row['run_id'], row['sample'])
        splice_vals.append(sp)
        short_vals.append(sh)
    sc_runs['annotated_splice_pct'] = splice_vals
    sc_runs['pct_too_short'] = short_vals

    has_data = sc_runs['annotated_splice_pct'].notna().any() or sc_runs['pct_too_short'].notna().any()
    if has_data:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('Annotated splice % (higher = better)',
                            '% reads too short (lower = better)'),
            horizontal_spacing=0.12
        )
        fig.add_trace(go.Bar(
            x=sc_runs['sample'], y=sc_runs['annotated_splice_pct'],
            marker_color='#2d6a4f'
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=sc_runs['sample'], y=sc_runs['pct_too_short'],
            marker_color='#e63946'
        ), row=1, col=2)
        fig.update_xaxes(tickangle=45)
        fig.update_layout(height=400, showlegend=False, title='STAR alignment QC — sc/bulk runs')
        fig.show()
    else:
        print('STAR log files not found (intermediates may have been deleted).')
    display(sc_runs[['run_id', 'sample', 'mapping_rate', 'annotated_splice_pct', 'pct_too_short']])

## Raw SQL explorer

In [ ]:
sql_box = widgets.Textarea(
    value='SELECT s.name, r.run_id, r.mapping_rate, r.total_filt\nFROM runs r JOIN samples s ON r.sample_id = s.sample_id\nORDER BY r.run_date DESC;',
    layout=widgets.Layout(width='100%', height='100px')
)
run_btn = widgets.Button(description='Run query', button_style='primary')

def on_run_query(btn):
    clear_output(wait=False)
    display(sql_box, run_btn)
    try:
        display(query(sql_box.value))
    except Exception as e:
        print(f'Error: {e}')

run_btn.on_click(on_run_query)
display(sql_box, run_btn)